# MSA-Free Folding & Inverse-Folding Round-Trip

**BioPipelines example** — a closed self-consistency loop run entirely MSA-free. ESMFold folds a sequence, two inverse folders (Frame2Seq and ProteinMPNN) redesign it, ESMFold re-folds the designs, and ConformationalChange measures how closely the redesigned sequences recover the original fold — a head-to-head test of the two inverse folders.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[colab]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines import ESMFold, Frame2Seq, ProteinMPNN

with Pipeline(project="Setup", job="InstallTools"):
    ESMFold.install()
    Frame2Seq.install()
    ProteinMPNN.install()   # SE3nv; provides backbones to inverse-fold

## Cell 3: MSA-Free Round-Trip Pipeline

A full fold → inverse-fold → re-fold loop on the villin headpiece (1VII, a fast-folding benchmark), with no MSA anywhere:

1. **ESMFold** predicts the structure directly from sequence — the reference model (no MSA, seconds per model).
2. **Frame2Seq** (MSA-free structure-conditioned LM) and **ProteinMPNN** (the standard inverse folder) each redesign that structure — 4 sequences apiece, from identical input.
3. **ESMFold** re-folds both sets of designs.
4. **ConformationalChange** measures each refolded design against the *original* ESMFold model (Cα RMSD). Low RMSD = the redesigned sequence recovers the fold — i.e. the design is self-consistent.
5. A **Plot** compares the two inverse folders head-to-head: whose sequences refold closest to the original?

Because ESMFold and Frame2Seq are both single-sequence, the whole loop runs offline on one GPU — ideal for tight design iterations. (Frame2Seq and ProteinMPNN also report a native-sequence recovery column for a sequence-level view.)

In [ ]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines import PDB, ESMFold, Frame2Seq, ProteinMPNN, ConformationalChange, Plot

with Pipeline(project="VillinHeadpiece", job="MSAFreeRoundTrip"):
    Resources(gpu="A100", time="2:00:00", memory="16GB")
    villin = PDB("1VII", chain="A")

    # Fold from sequence with no MSA -> the reference model.
    esm = ESMFold(sequences=villin, num_recycles=4)

    # Inverse-fold the predicted structure two ways (same input, 4 designs each).
    frame2seq = Frame2Seq(structures=esm, num_sequences=4)
    mpnn      = ProteinMPNN(structures=esm, num_sequences=4)

    # Re-fold both design sets (still MSA-free).
    refold_f2s  = ESMFold(sequences=frame2seq, num_recycles=4)
    refold_mpnn = ESMFold(sequences=mpnn, num_recycles=4)

    # Close the loop: how close do the redesigned sequences refold to the original?
    drift_f2s  = ConformationalChange(reference_structures=esm,
                                      target_structures=refold_f2s, atoms="CA")
    drift_mpnn = ConformationalChange(reference_structures=esm,
                                      target_structures=refold_mpnn, atoms="CA")

    # Head-to-head: which inverse folder yields the more self-consistent designs?
    Plot(Plot.Column(data=[drift_f2s.tables.changes, drift_mpnn.tables.changes],
                     y="RMSD",
                     labels=["Frame2Seq", "ProteinMPNN"],
                     style="box",
                     title="Re-fold RMSD to original (lower = more self-consistent)",
                     ylabel="Cα RMSD [Angstrom]"))
drift_f2s.tables.changes